[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/060.%20The%20Warehouse%20Slotting%20Optimization%20Problem/P49-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 49. The Causal and Regression-Based Forecasting Problem

## Tier 3: Genetic Algorithm for Feature Selection and Elastic Net Tuning

This notebook uses a genetic algorithm (GA) to search for a predictor subset and Elastic Net settings that achieve a low cross-validated error, with a small additional penalty for using more predictors. The input is a synthetic port-throughput dataset containing 120 observations, 12 candidate predictors, and one target. The outputs are the best chromosome found during the search, its fitness history, the selected predictors and hyperparameters, and diagnostic model results.

The GA is a stochastic search heuristic: it evolves a population of candidate solutions using selection, crossover, mutation, and elitism. It can explore many parts of the search space, but it does not guarantee a global optimum. This notebook's implementation uses a single combined fitness value, so it is a single-objective search rather than a formal multi-objective optimization.

### Scope and modeling assumptions

- The data are synthetic, and the data-generation process is known. The six predictors used to construct the signal are labeled as signal predictors for the later selection diagnostic; this label does not establish causal effects from observational data.
- The fitted model uses selected predictors as main effects. Although the synthetic target includes a trade-volume-by-GDP-growth interaction, the chromosome does not encode interaction terms.
- Predictors are passed to Elastic Net on their original scales. Because regularization is scale-sensitive, the numerical meaning of its penalty depends on predictor units.
- Five-fold cross-validation estimates prediction error on held-out folds of this dataset. It does not by itself establish performance on future data or prove that selected predictors are causal.
- The fixed complexity penalty is a heuristic added to mean squared error (MSE). It is not a scale-free information criterion, and its influence depends on the target's units and the chosen coefficient.

### Search representation and fitness

A chromosome contains one binary gene per predictor, followed by three real-valued genes: a ridge-like alpha, a lasso-like alpha, and an Elastic Net mixing ratio. A binary gene of 1 includes its predictor; 0 excludes it. The two alpha genes are blended into one Elastic Net `alpha`, so they are not two independently applied penalties.

For a nonempty selected feature set \(S\), the implementation minimizes

$$F(S, a_r, a_l, \rho) = \frac{1}{5}\sum_{k=1}^{5}\operatorname{MSE}_k\!\left(\operatorname{ElasticNet}(X_{S}, y; a, \rho)\right) + 0.1|S|,$$

where \(\operatorname{MSE}_k\) is the mean squared prediction error on cross-validation fold \(k\), \(a = a_r(1-\rho) + a_l\rho\), and \(\rho\) is the Elastic Net `l1_ratio`. Here \(S\) is the set of selected predictor indices, \(a_r\) and \(a_l\) are the two alpha genes, and \(|S|\) is the number of selected predictors. The code returns a large penalty for an empty feature set instead of fitting a model. Lower fitness is preferred.

The fitness function evaluates each candidate on the same five folds. Tournament selection samples candidate parents and chooses the lowest-fitness participant; crossover recombines their genes; mutation flips binary genes or perturbs real genes; and elitism carries the best candidates into the next generation. The best-so-far fitness is retained across generations.

If \(P\) is the population size, \(G\) the number of generations, and \(K=5\) the number of folds, fitness evaluation requires approximately \(GPK\) model fits, in addition to the final model fit and diagnostic calculations. The cost of each fit depends on the number of observations and selected predictors. The GA is stochastic, and the recorded best solution is the best one encountered in this run, not a proof of global optimality.

### How to read the results

- The best-fitness trace records the best candidate encountered up to each generation; the average-fitness trace summarizes the current population. A flat best-fitness trace means no better candidate was found during that interval, not that all possible solutions have been searched.
- Selected predictors and the blended `alpha` and `l1_ratio` describe the candidate used for the final fit. The individual alpha genes are also reported to make the chromosome representation visible.
- The final training-set MSE and \(R^2\) are in-sample descriptive metrics. They are not the cross-validated MSE used in the GA fitness and should not be interpreted as an independent estimate of future predictive performance.
- The feature-selection diagnostic compares the chosen subset with the known synthetic signal/noise labels. It evaluates recovery in this constructed example, not causal discovery in general.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# A consistent visual style makes the diagnostic plots easier to compare.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
# Generate a reproducible synthetic dataset for the feature-selection example.
np.random.seed(42)

n_samples = 120

# These six variables contribute signal to the synthetic target.
trade_volume = np.random.normal(100, 15, n_samples)
gdp_growth = np.random.normal(2.5, 0.8, n_samples)
fuel_price = np.random.normal(85, 10, n_samples)
seasonal_peak = np.random.binomial(1, 0.3, n_samples)
ship_density = np.random.normal(25, 5, n_samples)
exchange_rate = np.random.normal(1.2, 0.15, n_samples)

# These variables are generated as noise candidates and do not enter the target equation.
random_noise_1 = np.random.normal(0, 1, n_samples)
random_noise_2 = np.random.normal(0, 1, n_samples)
random_noise_3 = np.random.normal(0, 1, n_samples)
competition = np.random.normal(50, 10, n_samples)
weather_index = np.random.normal(100, 20, n_samples)
policy_dummy = np.random.binomial(1, 0.2, n_samples)

# The target includes an interaction term, while the fitted Elastic Net uses main effects only.
true_throughput = (
    2000
    + 15 * trade_volume
    + 300 * gdp_growth
    + 8 * fuel_price
    + 250 * seasonal_peak
    + 20 * ship_density
    + 150 * exchange_rate
    + 50 * (trade_volume * gdp_growth) / 100
    + np.random.normal(0, 250, n_samples)
)

data = pd.DataFrame(
    {
        "trade_volume": trade_volume,
        "gdp_growth": gdp_growth,
        "fuel_price": fuel_price,
        "seasonal_peak": seasonal_peak,
        "ship_density": ship_density,
        "exchange_rate": exchange_rate,
        "random_noise_1": random_noise_1,
        "random_noise_2": random_noise_2,
        "random_noise_3": random_noise_3,
        "competition": competition,
        "weather_index": weather_index,
        "policy_dummy": policy_dummy,
        "throughput": true_throughput,
    }
)

predictor_columns = [column for column in data.columns if column != "throughput"]

print("Synthetic port-throughput dataset")
print(f"Shape: {data.shape}")
print(f"Number of candidate predictors: {len(predictor_columns)}")
print("\nPredictor variables:")
for index, column in enumerate(predictor_columns, start=1):
    print(f"  {index:2d}. {column}")

print("\nDataset statistics:")
print(data.describe().round(2))

## Genetic algorithm implementation

The class below keeps the search steps explicit. `initialize_population` creates candidate chromosomes; `evaluate_fitness` translates each chromosome into a feature subset and Elastic Net configuration; `tournament_selection`, `crossover`, and `mutate` create the next generation; and `evolve` coordinates these steps while recording fitness history.

The fitness is evaluated by five-fold cross-validation. For regression, scikit-learn's integer `cv=5` uses five non-shuffled folds by default. The `predictor_names` argument is used only for readable progress messages; the algorithm does not read predictor names or data from hidden global variables.

In [ ]:
class GeneticAlgorithmRegressor:
    """Search feature subsets and Elastic Net settings with a genetic algorithm."""

    def __init__(
        self,
        n_predictors,
        population_size=40,
        crossover_rate=0.8,
        mutation_rate=0.15,
        elite_size=4,
        max_generations=100,
    ):
        self.n_predictors = n_predictors
        self.population_size = population_size
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size
        self.max_generations = max_generations

        # Each chromosome stores binary feature genes and three real-valued genes.
        self.chromosome_length = n_predictors + 3

    def initialize_population(self):
        """Create a population with binary feature genes and bounded real genes."""
        population = []

        for _ in range(self.population_size):
            binary_genes = np.random.binomial(1, 0.5, self.n_predictors)
            ridge_alpha = np.random.uniform(0.01, 10.0)
            lasso_alpha = np.random.uniform(0.01, 10.0)
            elastic_ratio = np.random.uniform(0.0, 1.0)

            chromosome = np.concatenate(
                [binary_genes, [ridge_alpha, lasso_alpha, elastic_ratio]]
            )
            population.append(chromosome)

        return np.array(population)

    def evaluate_fitness(self, chromosome, X, y):
        """Return cross-validated MSE plus the specified feature-count penalty."""
        feature_mask = chromosome[: self.n_predictors].astype(bool)
        ridge_alpha = chromosome[self.n_predictors]
        lasso_alpha = chromosome[self.n_predictors + 1]
        elastic_ratio = chromosome[self.n_predictors + 2]

        # An empty subset cannot be fitted by this model, so it receives a poor score.
        if not np.any(feature_mask):
            return 1e6

        X_selected = X[:, feature_mask]
        alpha = ridge_alpha * (1 - elastic_ratio) + lasso_alpha * elastic_ratio

        model = ElasticNet(
            alpha=alpha,
            l1_ratio=elastic_ratio,
            max_iter=1000,
            random_state=42,
        )
        cv_scores = cross_val_score(
            model,
            X_selected,
            y,
            cv=5,
            scoring="neg_mean_squared_error",
        )
        mse = -cv_scores.mean()

        # This fixed term expresses a preference for smaller subsets in the stated fitness.
        n_features = np.sum(feature_mask)
        complexity_penalty = 0.1 * n_features
        return mse + complexity_penalty

    def tournament_selection(self, population, fitness_values, tournament_size=3):
        """Choose one low-fitness candidate from each randomly sampled tournament."""
        selected = []

        for _ in range(len(population)):
            tournament_indices = np.random.choice(
                len(population), tournament_size, replace=False
            )
            tournament_fitness = fitness_values[tournament_indices]
            winner_idx = tournament_indices[np.argmin(tournament_fitness)]
            selected.append(population[winner_idx].copy())

        return np.array(selected)

    def crossover(self, parent1, parent2):
        """Recombine binary genes and blend real-valued genes."""
        if np.random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()

        child1 = parent1.copy()
        child2 = parent2.copy()

        for index in range(len(parent1)):
            if index < self.n_predictors:
                if np.random.random() < 0.5:
                    child1[index], child2[index] = parent2[index], parent1[index]
            elif np.random.random() < 0.5:
                blend = np.random.random()
                child1[index] = blend * parent1[index] + (1 - blend) * parent2[index]
                child2[index] = (1 - blend) * parent1[index] + blend * parent2[index]

        return child1, child2

    def mutate(self, chromosome):
        """Flip selected binary genes or perturb real genes within their bounds."""
        mutated = chromosome.copy()

        for index in range(len(chromosome)):
            if np.random.random() >= self.mutation_rate:
                continue

            if index < self.n_predictors:
                mutated[index] = 1 - mutated[index]
            elif index == self.n_predictors:
                mutated[index] = np.clip(
                    mutated[index] + np.random.normal(0, 1), 0.01, 10.0
                )
            elif index == self.n_predictors + 1:
                mutated[index] = np.clip(
                    mutated[index] + np.random.normal(0, 1), 0.01, 10.0
                )
            else:
                mutated[index] = np.clip(
                    mutated[index] + np.random.normal(0, 0.2), 0.0, 1.0
                )

        return mutated

    def evolve(self, X, y, predictor_names=None, verbose=True):
        """Run the population search and return the best candidate encountered."""
        if X.shape[1] != self.n_predictors:
            raise ValueError("X must have one column for each configured predictor.")
        if predictor_names is not None and len(predictor_names) != self.n_predictors:
            raise ValueError("predictor_names must match the number of predictors.")

        population = self.initialize_population()
        best_fitness_history = []
        avg_fitness_history = []
        fitness_std_history = []
        best_chromosome = None
        best_fitness = float("inf")

        if verbose:
            print("Genetic algorithm evolution progress:")
            print("=" * 60)

        for generation in range(self.max_generations):
            fitness_values = np.array(
                [self.evaluate_fitness(chromosome, X, y) for chromosome in population]
            )

            current_best_idx = np.argmin(fitness_values)
            current_best_fitness = fitness_values[current_best_idx]
            if current_best_fitness < best_fitness:
                best_fitness = current_best_fitness
                best_chromosome = population[current_best_idx].copy()

            best_fitness_history.append(best_fitness)
            avg_fitness_history.append(np.mean(fitness_values))
            fitness_std_history.append(np.std(fitness_values))

            should_report = (
                generation == 0
                or generation % 20 == 19
                or generation == self.max_generations - 1
            )
            if verbose and should_report:
                feature_mask = best_chromosome[: self.n_predictors].astype(bool)
                selected_indices = np.where(feature_mask)[0]
                ridge_alpha = best_chromosome[self.n_predictors]
                lasso_alpha = best_chromosome[self.n_predictors + 1]
                elastic_ratio = best_chromosome[self.n_predictors + 2]
                if predictor_names is None:
                    selected_names = selected_indices.tolist()
                else:
                    selected_names = [predictor_names[i] for i in selected_indices]

                print(f"Generation {generation}: Best fitness = {best_fitness:.4f}")
                print(f"  Selected predictors: {selected_names}")
                print(
                    f"  Ridge-like alpha gene: {ridge_alpha:.4f}, "
                    f"lasso-like alpha gene: {lasso_alpha:.4f}, "
                    f"L1 ratio: {elastic_ratio:.4f}"
                )
                print()

            selected_population = self.tournament_selection(population, fitness_values)
            new_population = []

            # Elitism preserves the strongest current candidates in the next generation.
            elite_indices = np.argsort(fitness_values)[: self.elite_size]
            for index in elite_indices:
                new_population.append(population[index].copy())

            while len(new_population) < self.population_size:
                parent_indices = np.random.choice(len(selected_population), 2, replace=False)
                parent1, parent2 = selected_population[parent_indices]
                child1, child2 = self.crossover(parent1, parent2)
                new_population.extend([self.mutate(child1), self.mutate(child2)])

            population = np.array(new_population[: self.population_size])

        return {
            "best_chromosome": best_chromosome,
            "best_fitness": best_fitness,
            "fitness_history": best_fitness_history,
            "avg_fitness_history": avg_fitness_history,
            "fitness_std_history": fitness_std_history,
        }

## Prepare the data and run the search

The feature matrix \(X\) contains the 12 candidate predictors, and the target vector \(y\) contains throughput. The chromosome therefore has 15 genes: 12 binary inclusion indicators and three real-valued hyperparameter genes. The fixed population and generation settings below define the search budget; the result is stochastic even though the synthetic-data seed is fixed.

In [ ]:
X = data[predictor_columns].to_numpy()
y = data["throughput"].to_numpy()

print("Data prepared for the genetic algorithm:")
print(f"Samples: {X.shape[0]}")
print(f"Predictors: {X.shape[1]}")
print(
    f"Chromosome length: {X.shape[1] + 3} "
    f"({X.shape[1]} binary genes + 3 real-valued genes)"
)

ga = GeneticAlgorithmRegressor(
    n_predictors=len(predictor_columns),
    population_size=40,
    crossover_rate=0.8,
    mutation_rate=0.15,
    elite_size=4,
    max_generations=100,
)

ga_results = ga.evolve(X, y, predictor_names=predictor_columns, verbose=True)

## Interpret and fit the best candidate

The search returns the best chromosome encountered, not a guarantee that no better chromosome exists. The final Elastic Net fit uses the chromosome's selected predictors and blended alpha. Its reported training metrics describe fit to the same observations used throughout the search; the cross-validated fitness remains the relevant quantity for comparing candidates under the specified objective.

In [ ]:
best_chromosome = ga_results["best_chromosome"]
best_fitness = ga_results["best_fitness"]

feature_mask = best_chromosome[: len(predictor_columns)].astype(bool)
selected_features = [
    name for name, selected in zip(predictor_columns, feature_mask) if selected
]
if not selected_features:
    raise RuntimeError("The best chromosome contains no selected predictors.")

ridge_alpha = best_chromosome[len(predictor_columns)]
lasso_alpha = best_chromosome[len(predictor_columns) + 1]
elastic_ratio = best_chromosome[len(predictor_columns) + 2]
final_alpha = ridge_alpha * (1 - elastic_ratio) + lasso_alpha * elastic_ratio

print("\n" + "=" * 60)
print("GENETIC ALGORITHM RESULTS")
print("=" * 60)
print(f"\nSelected predictors: {selected_features}")
print(f"Number selected: {len(selected_features)}/{len(predictor_columns)}")
print(f"Ridge-like alpha gene: {ridge_alpha:.4f}")
print(f"Lasso-like alpha gene: {lasso_alpha:.4f}")
print(f"Blended Elastic Net alpha: {final_alpha:.4f}")
print(f"Elastic Net L1 ratio: {elastic_ratio:.4f}")
print(f"Best cross-validated fitness: {best_fitness:.4f}")

X_selected = X[:, feature_mask]
final_model = ElasticNet(
    alpha=final_alpha,
    l1_ratio=elastic_ratio,
    max_iter=1000,
    random_state=42,
)
final_model.fit(X_selected, y)

y_pred = final_model.predict(X_selected)
final_mse = mean_squared_error(y, y_pred)
final_r2 = r2_score(y, y_pred)

print("\nFinal model training-set metrics:")
print(f"MSE: {final_mse:.2f}")
print(f"R-squared: {final_r2:.4f}")
print("Model coefficients:")
for feature, coefficient in zip(selected_features, final_model.coef_):
    print(f"  {feature}: {coefficient:.4f}")
print(f"Intercept: {final_model.intercept_:.4f}")

assert len(selected_features) == int(feature_mask.sum())
assert np.isclose(final_alpha, ridge_alpha * (1 - elastic_ratio) + lasso_alpha * elastic_ratio)

## Visual diagnostics

The panels show fitness across generations, the final inclusion decision for every candidate predictor, the final hyperparameter genes, and an in-sample comparison between the all-predictor baseline and the selected model. The two models in the last panel are fitted and scored on the same full dataset, so that panel is descriptive rather than an independent predictive-performance comparison. MSE, \(R^2\), and feature ratio have different scales; the plotted bars should be read by their labels and value annotations, not as directly comparable units.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Genetic Algorithm Diagnostics for Regression Model Selection",
    fontsize=16,
    fontweight="bold",
)

# The best-so-far trace should not increase because the best candidate is retained.
generations = np.arange(len(ga_results["fitness_history"]))
axes[0, 0].plot(
    generations,
    ga_results["fitness_history"],
    linewidth=2,
    label="Best-so-far fitness",
)
axes[0, 0].plot(
    generations,
    ga_results["avg_fitness_history"],
    linestyle="--",
    linewidth=2,
    label="Population mean fitness",
)
axes[0, 0].set_xlabel("Generation")
axes[0, 0].set_ylabel("Fitness (cross-validated MSE + penalty)")
axes[0, 0].set_title("Fitness by Generation")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Showing every candidate makes both inclusion and exclusion decisions visible.
selection_status = feature_mask.astype(int)
axes[0, 1].bar(
    np.arange(len(predictor_columns)),
    selection_status,
    color=["seagreen" if value else "lightgray" for value in selection_status],
    edgecolor="black",
)
axes[0, 1].set_xlabel("Candidate predictor")
axes[0, 1].set_ylabel("Selected (1) or excluded (0)")
axes[0, 1].set_title("Final Feature-Selection Decision")
axes[0, 1].set_xticks(np.arange(len(predictor_columns)))
axes[0, 1].set_xticklabels(predictor_columns, rotation=45, ha="right")
axes[0, 1].set_ylim(-0.05, 1.15)
axes[0, 1].grid(True, axis="y", alpha=0.3)

# The plotted alpha genes are chromosome values; the fitted alpha is their blend.
axes[1, 0].scatter(
    [ridge_alpha],
    [elastic_ratio],
    s=200,
    color="red",
    marker="*",
    edgecolors="black",
    linewidth=1.5,
    label="Best chromosome",
)
axes[1, 0].set_xlabel("Ridge-like alpha gene")
axes[1, 0].set_ylabel("Elastic Net L1 ratio")
axes[1, 0].set_title("Selected Hyperparameter Genes")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim(0, 10)
axes[1, 0].set_ylim(0, 1)

# This baseline uses all predictors and fixed Elastic Net settings for context.
baseline_model = ElasticNet(
    alpha=1.0,
    l1_ratio=0.5,
    max_iter=1000,
    random_state=42,
)
baseline_model.fit(X, y)
baseline_pred = baseline_model.predict(X)
baseline_mse = mean_squared_error(y, baseline_pred)
baseline_r2 = r2_score(y, baseline_pred)

model_names = ["All predictors", "GA-selected"]
mse_values = [baseline_mse, final_mse]
r2_values = [baseline_r2, final_r2]
feature_ratios = [1.0, len(selected_features) / len(predictor_columns)]
x_positions = np.arange(len(model_names))
bar_width = 0.25

bars_mse = axes[1, 1].bar(
    x_positions - bar_width,
    mse_values,
    bar_width,
    label="Training MSE",
    color="orange",
    alpha=0.7,
)
bars_r2 = axes[1, 1].bar(
    x_positions,
    r2_values,
    bar_width,
    label="Training R-squared",
    color="royalblue",
    alpha=0.7,
)
bars_features = axes[1, 1].bar(
    x_positions + bar_width,
    feature_ratios,
    bar_width,
    label="Feature ratio",
    color="seagreen",
    alpha=0.7,
)
axes[1, 1].set_xlabel("Model")
axes[1, 1].set_ylabel("Metric value (units differ by metric)")
axes[1, 1].set_title("In-Sample Model Summary")
axes[1, 1].set_xticks(x_positions)
axes[1, 1].set_xticklabels(model_names)
axes[1, 1].legend()
axes[1, 1].grid(True, axis="y", alpha=0.3)

for bars, values in [
    (bars_mse, mse_values),
    (bars_r2, r2_values),
    (bars_features, feature_ratios),
]:
    for bar, value in zip(bars, values):
        height = bar.get_height()
        offset = max(abs(height) * 0.01, 0.01)
        axes[1, 1].text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

## Feature-selection and convergence diagnostics

Because the synthetic target is constructed from a known set of signal predictors, selection can be summarized with precision, recall, and accuracy against those labels. These quantities assess recovery of the data-generation labels in this example; they are not general measures of causal discovery. Convergence diagnostics summarize observed search behavior, not proof of global optimality. The standard deviation below measures fitness spread within each generation, not diversity of chromosome genotypes.

In [ ]:
print("\n" + "=" * 60)
print("FEATURE-SELECTION ANALYSIS")
print("=" * 60)

true_causal = [
    "trade_volume",
    "gdp_growth",
    "fuel_price",
    "seasonal_peak",
    "ship_density",
    "exchange_rate",
]
true_noise = [
    "random_noise_1",
    "random_noise_2",
    "random_noise_3",
    "competition",
    "weather_index",
    "policy_dummy",
]

print("Signal predictors in the synthetic target equation:")
for variable in true_causal:
    status = "Selected" if variable in selected_features else "Not selected"
    print(f"  {variable}: {status}")

print("\nNoise candidates absent from the synthetic target equation:")
for variable in true_noise:
    status = "Excluded" if variable not in selected_features else "Selected"
    print(f"  {variable}: {status}")

true_positives = sum(variable in selected_features for variable in true_causal)
false_positives = sum(variable in selected_features for variable in true_noise)
false_negatives = sum(variable not in selected_features for variable in true_causal)
true_negatives = sum(variable not in selected_features for variable in true_noise)

precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives
    else 0.0
)
recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives
    else 0.0
)
accuracy = (true_positives + true_negatives) / len(predictor_columns)

print("\nSelection metrics against the synthetic labels:")
print(f"  Precision: {precision:.3f}")
print(f"  Recall: {recall:.3f}")
print(f"  Accuracy: {accuracy:.3f}")

fitness_history = ga_results["fitness_history"]
if len(fitness_history) >= 20:
    reference_fitness = fitness_history[-20]
    final_best_fitness = fitness_history[-1]
    if not np.isclose(reference_fitness, 0.0):
        recent_improvement = (
            reference_fitness - final_best_fitness
        ) / abs(reference_fitness)
        print("\nBest-so-far fitness change over the final 20-generation interval:")
        print(f"  Relative improvement: {recent_improvement * 100:.2f}%")
        if recent_improvement < 0.01:
            print("  Little additional improvement was observed in this interval.")
        else:
            print("  The best candidate improved during this interval.")

recent_population_spread = ga_results["fitness_std_history"][-10:]
print(
    "\nMean within-generation fitness standard deviation over the final "
    f"{len(recent_population_spread)} generations: "
    f"{np.mean(recent_population_spread):.4f}"
)

## Reading the diagnostics together

The GA's cross-validated fitness, the final model's training metrics, and the synthetic-label selection metrics answer different questions. Fitness is the objective the search minimizes; training MSE and \(R^2\) describe fit on the complete dataset; and precision, recall, and accuracy describe agreement with the known synthetic feature labels. These values should not be treated as interchangeable or as evidence that the fitted model has discovered causal relationships.

The search is computationally more expensive than evaluating a single model because each candidate requires five model fits and the population is reevaluated across generations. Its benefit here is the joint search over binary feature decisions and real-valued Elastic Net settings. Its limitations include stochastic search, sensitivity to the fitness definition and predictor scales, and the absence of interaction genes despite an interaction in the synthetic target.

In [ ]:
print("\n" + "=" * 60)
print("FITNESS AND MODEL-METRIC INTERPRETATION")
print("=" * 60)
print(f"Best cross-validated fitness (MSE plus feature penalty): {best_fitness:.4f}")
print(f"Final model training MSE: {final_mse:.2f}")
print(f"Final model training R-squared: {final_r2:.4f}")
print(
    "The cross-validated fitness and training MSE use different evaluation procedures; "
    "they are reported separately and are not direct substitutes for one another."
)
print(
    f"Selected {len(selected_features)} of {len(predictor_columns)} candidate predictors."
)

## Why this tier differs from a greedy search

A population-based search maintains several candidate solutions at once and recombines their feature and hyperparameter genes. This makes joint exploration possible, while a greedy procedure typically makes sequential choices. The GA can therefore encounter combinations that a particular greedy path would not visit, but it does not guarantee that it will find a better solution or avoid every local optimum.

### Strengths and limitations

**Strengths**
- Binary and real-valued genes represent feature inclusion and model settings in one chromosome.
- Selection, crossover, mutation, and elitism provide a structured way to explore and retain candidate solutions.
- The recorded fitness history makes progress during the specified search visible.

**Limitations**
- Repeated cross-validation makes the search computationally expensive.
- Population size, mutation rate, crossover rate, and generation count affect search behavior.
- A stochastic run does not certify a global optimum, and its result can depend on the random-number state.
- The fixed complexity penalty and unscaled predictors affect how the fitness and regularization behave.

### When this approach may be useful

This approach may be useful when predictor inclusion and model settings need to be searched jointly and the available computational budget permits repeated model fitting. For a real forecasting application, the data-splitting design, predictor scaling, time dependence, and the interpretation of the complexity penalty should reflect the problem's requirements.